In [ ]:
pip install git+https://github.com/huggingface/transformers.git

tiny genimage

In [ ]:
# -*- coding: utf-8 -*-
import os
from PIL import Image
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from transformers import pipeline
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')


TINY_PATH = "/content/drive/MyDrive/my_ai_research/tiny_genimage"


models = {
    "AIorNot-SigLIP2": "prithivMLmods/AIorNot-SigLIP2",
    "AI-image-detector": "umm-maybe/AI-image-detector",
    "AI-vs-Deepfake-vs-Real-Siglip2": "prithivMLmods/AI-vs-Deepfake-vs-Real-Siglip2"
}


classifiers = {}
for name, model_id in models.items():
    print(f"Loading {name}...")
    classifiers[name] = pipeline("image-classification", model=model_id)

def predict(model_name, img_path):
    try:
        result = classifiers[model_name](img_path)[0]
        label = result['label'].lower()
        if "AI-vs-Deepfake" in model_name:
            return 1 if label in ['ai', 'deepfake'] else 0
        else:
            return 1 if label == 'fake' else 0
    except:
        return 0

results = []
for gen_name in os.listdir(TINY_PATH):
    gen_dir = os.path.join(TINY_PATH, gen_name)
    if not os.path.isdir(gen_dir):
        continue
    real_dir = os.path.join(gen_dir, "train", "nature")
    fake_dir = os.path.join(gen_dir, "train", "ai")
    if not os.path.exists(real_dir) or not os.path.exists(fake_dir):
        print(f"Skipping {gen_name}: no train/nature or train/ai")
        continue

    real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))][:100]
    fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))][:100]
    print(f"\n{gen_name}: {len(real_files)} real, {len(fake_files)} fake")

    for model_name in models.keys():
        preds = []
        for path in tqdm(real_files + fake_files, desc=f"{model_name} on {gen_name}"):
            preds.append(predict(model_name, path))
        true = [0]*len(real_files) + [1]*len(fake_files)
        acc = accuracy_score(true, preds)
        results.append({"Generator": gen_name, "Model": model_name, "Accuracy": acc})
        print(f"  {model_name}: {acc:.2%}")


df = pd.DataFrame(results)
pivot = df.pivot(index="Generator", columns="Model", values="Accuracy")
print("\n=== Tiny GenImage Results ===")
print(pivot)

out_dir = "/content/drive/MyDrive/my_ai_research/detection_results"
os.makedirs(out_dir, exist_ok=True)
pivot.to_csv(os.path.join(out_dir, "tiny_genimage_results.csv"))
print(f"Saved to {out_dir}/tiny_genimage_results.csv")